In [ ]:
from google.colab import userdata
API_KEY = userdata.get('GNANIAPIKEY')

In [ ]:
!pip install fastapi uvicorn python-multipart nest-asyncio pyngrok

In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [ ]:
import requests
import json
from google.colab import files, userdata
from IPython.display import Audio, display

# 🔑 Load your API key securely
API_KEY = userdata.get('GNANIAPIKEY')
BASE_URL = "https://api.vachana.ai"

HEADERS_KEY = {
    "X-API-Key-ID": API_KEY
}

# ------------------------------------------------------------------
# PART 1: SPEECH-TO-TEXT (Prisma v2.5)
# ------------------------------------------------------------------
def transcribe_audio_file(file_path, language_code="hi-IN"):
    url = f"{BASE_URL}/stt/v3"

    with open(file_path, "rb") as audio_file:
        files_payload = {"audio_file": (file_path, audio_file)}
        data_payload = {
            "language_code": language_code,
            "preferred_language": language_code,
            "format": "transcribe",
            "itn_native_numerals": "true"
        }

        print("⏳ Transcribing patient audio with Prisma v2.5...")
        response = requests.post(url, headers=HEADERS_KEY, data=data_payload, files=files_payload)

    if response.status_code == 200:
        result = response.json()
        transcript = result.get("transcript") or result.get("text") or str(result)
        return transcript
    else:
        print(f"❌ STT Error {response.status_code}: {response.text}")
        return None



In [ ]:
# PART 2: ADVANCED CLINICAL TRIAGE & SOLUTIONS ENGINE
# ------------------------------------------------------------------
def evaluate_symptoms_and_solutions(transcript):
    if not transcript:
        return "UNKNOWN", "विवरण नहीं मिला।", []
    t_lower = transcript.lower()

    # 1. RED TIER: Critical Emergencies (Chest pain, breathing difficulty)
    if any(w in t_lower for w in ["chest pain", "breathlessness", "unconscious", "saans", "छाती में दर्द", "सांस फूलना"]):
        return "RED (Emergency)", "चेतावनी: गंभीर लक्षण पाए गए हैं। तुरंत नजदीकी अस्पताल जाएं।", ["Call 108 emergency ambulance immediately.", "Keep the patient calm and seated upright."]

    # 2. EYE PAIN / VISION ISSUES (Specific to your audio3)
    elif any(w in t_lower for w in ["eye", "आँख", "लेफ्ट आई", "right eye", "आंख"]):
        return "YELLOW (Moderate)", "आपको आँख में दर्द या समस्या है। अपनी आँख को बार-बार न छुएं और तुरंत नेत्र विशेषज्ञ से मिलें।", ["Wash eyes gently with clean, lukewarm water.", "Avoid rubbing or straining your eyes with screens.", "Consult an ophthalmologist (eye specialist) within 24 hours."]

    # 3. HAND / ARM PAIN (Specific to your audio3)
    elif any(w in t_lower for w in ["hand", "हाथ", "arm", "उंगली"]):
        return "YELLOW (Moderate)", "आपको हाथ या बाजू में दर्द की शिकायत है। उस हाथ को आराम दें और ज्यादा वजन न उठाएं।", ["Rest the affected arm and apply a soft bandage if needed.", "Avoid lifting heavy objects.", "Consult a doctor if swelling or numbness occurs."]

    # 4. HEADACHE / SIR DARD
    elif any(w in t_lower for w in ["sir dard", "headache", "sar dard", "सर दर्द", "सिर में दर्द"]):
        return "YELLOW (Moderate)", "आपको सिर दर्द की शिकायत है। शांत और अंधेरे कमरे में आराम करें।", ["Apply a cold compress on your forehead.", "Stay hydrated and avoid bright screens or loud noises.", "Consult a doctor if headache persists."]

    # 5. LEG / FOOT PAIN
    elif any(w in t_lower for w in ["पैर में दर्द", "leg pain", "foot pain", "घुटना"]):
        return "YELLOW (Moderate)", "आपको पैर में दर्द है। पैर को ऊंचा रखकर आराम करें।", ["Rest the affected leg and avoid walking long distances.", "Apply ice pack if there is minor swelling.", "Consult a physician if pain continues."]

    # 6. FEVER / BUKHAR
    elif any(w in t_lower for w in ["bukhar", "fever", "बुखार"]):
        return "YELLOW (Moderate)", "आपके लक्षणों के अनुसार आपको बुखार है। पर्याप्त पानी पिएं और आराम करें।", ["Monitor body temperature every 4-6 hours.", "Stay hydrated with oral rehydration salts (ORS).", "Consult a general physician within 24 hours."]

    # 7. STOMACH / DIGESTIVE
    elif any(w in t_lower for w in ["pet", "stomach", "vomit", "ulti", "पेट", "दस्त"]):
        return "YELLOW (Moderate)", "पेट की समस्या के लिए हल्का और ताजा भोजन लें तथा ओआरएस का घोल पिएं।", ["Consume light foods like khichdi and boiled water.", "Avoid spicy, oily, or outside food.", "Maintain strict hydration."]

    # 8. GENERAL / MILD FALLBACK
    else:
        return "GREEN (Mild)", "आपके लक्षण सामान्य प्रतीत होते हैं। पर्याप्त विश्राम करें और स्वास्थ्य केंद्र से परामर्श लें।", ["Get adequate rest for 1-2 days.", "Maintain proper hydration and healthy diet."]

# ------------------------------------------------------------------


In [ ]:
# PART 3: TEXT-TO-SPEECH (Timbre v2.5)
# ------------------------------------------------------------------
def synthesize_voice(text, language_code="hi-IN", output_filename="response.wav"):
    url = f"{BASE_URL}/api/v1/tts/inference"
    headers_json = {**HEADERS_KEY, "Content-Type": "application/json"}

    payload = {
        "text": text,
        "voice": "Nalini",
        "model": "timbre-v2.5",
        "language": language_code,
        "speed": 1.0,
        "audio_config": {
            "sample_rate": 24000,
            "container": "wav"
        }
    }

    response = requests.post(url, headers=headers_json, json=payload)

    if response.status_code == 200:
        with open(output_filename, "wb") as f:
            f.write(response.content)
        return output_filename
    else:
        print(f"❌ TTS Error: {response.status_code} - {response.text}")
        return None

# ------------------------------------------------------------------


In [ ]:
# RUNNABLE INTERACTIVE FLOW
# ------------------------------------------------------------------
def run_live_health_bot():
    print("📂 Please upload your patient audio file (.wav or .mp3):")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    audio_filename = list(uploaded.keys())[0]
    print(f"\n✅ File uploaded: {audio_filename}\n")

    # Transcribe speech
    patient_text = transcribe_audio_file(audio_filename, language_code="hi-IN")
    print(f"🗣️ Transcribed Patient Speech: {patient_text}\n")

    if not patient_text:
        return

    # Evaluate triage & get tailored solutions
    triage_level, response_text, actionable_solutions = evaluate_symptoms_and_solutions(patient_text)

    print(f"🏥 Assessed Triage Level: {triage_level}")
    print(f"💬 Spoken Advice to Patient: {response_text}\n")

    print("📋 Actionable Medical Solutions / Next Steps for Health Worker:")
    for i, sol in enumerate(actionable_solutions, 1):
        print(f"   {i}. {sol}")
    print("\n")

    # Generate speech for the advice
    output_audio = synthesize_voice(response_text, language_code="hi-IN")

    if output_audio:
        print("🔊 Playing Generated Voice Response:")
        display(Audio(output_audio, autoplay=True))

run_live_health_bot()

📂 Please upload your patient audio file (.wav or .mp3):


Saving audio3.wav to audio3 (2).wav

✅ File uploaded: audio3 (2).wav

⏳ Transcribing patient audio with Prisma v2.5...
🗣️ Transcribed Patient Speech: हेलो मैं बोल रहा था कि मेरे हाथ में बहुत दर्द है और मेरा वो लेफ्ट आई में बहुत दर्द हो रहा है तो क्या करना चाहिए मुझे

🏥 Assessed Triage Level: YELLOW (Moderate)
💬 Spoken Advice to Patient: आपको आँख में दर्द या समस्या है। अपनी आँख को बार-बार न छुएं और तुरंत नेत्र विशेषज्ञ से मिलें।

📋 Actionable Medical Solutions / Next Steps for Health Worker:
   1. Wash eyes gently with clean, lukewarm water.
   2. Avoid rubbing or straining your eyes with screens.
   3. Consult an ophthalmologist (eye specialist) within 24 hours.


🔊 Playing Generated Voice Response:
